<a href="https://colab.research.google.com/github/okilmaga-cyber/Lessons/blob/main/%D0%A3%D1%80%D0%BE%D0%BA10_%D1%81%D0%B0%D0%BC%D0%BE%D1%81%D1%82%D0%BE%D1%8F%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏭 Урок 10 — Самостоятельная работа: Pipeline и кросс-валидация

### Как работать с этим ноутбуком
1. **Читай** теорию, **запускай** код по порядку, **выполняй задания** 📝, **отвечай словами** ✍️.
2. К концу урока у тебя будет собственный **конвейер (Pipeline)** для машинного обучения.

> 🎯 **К концу урока ты сможешь:** собрать подготовку данных и модель в один `Pipeline`, обработать числа и категории через `ColumnTransformer`, честно оценить модель через кросс-валидацию и объяснить, что такое утечка данных.

> 🏭 **Аналогия:** Pipeline — это конвейер. Сырые данные входят с одного конца, предсказание выходит с другого. Шаги всегда в правильном порядке.

## Шаг 0 · Данные КАК ЕСТЬ
На прошлом уроке мы чистили данные руками. Сегодня — **не будем**. Оставим пропуски и текст как есть: всю подготовку сделает Pipeline.

Наша задача — только сказать, где **числа**, а где **категории**.

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split

df = sns.load_dataset('titanic')

num_features = ['age', 'fare', 'sibsp', 'parch']   # числовые признаки
cat_features = ['sex', 'pclass', 'embarked']         # категориальные признаки

X = df[num_features + cat_features]   # НЕ чистим руками!
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

X.head()   # обрати внимание: есть пропуски (NaN) и текст — это нормально

## Шаг 1 · Разная обработка для чисел и категорий
Нельзя масштабировать слово «female» и нельзя one-hot-кодировать возраст. Поэтому у нас **два мини-конвейера**:
- **Числа:** заполнить пропуски медианой → масштабировать.
- **Категории:** заполнить частым значением → one-hot (категории → столбцы 0/1).

Собираем их в `ColumnTransformer` — он направит каждую колонку в нужный конвейер.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Мини-конвейер для чисел
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),   # пропуски -> медиана
    ('scale',  StandardScaler()),                    # масштабирование
])

# Мини-конвейер для категорий
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),   # пропуски -> частое значение
    ('onehot', OneHotEncoder(handle_unknown='ignore')),    # категории -> 0/1
])

# Кто куда: числам — num_pipe, категориям — cat_pipe
preprocess = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features),
])
print('Препроцессор готов ✅ (пока ничего не обучали — только описали шаги)')

## Шаг 2 · Собираем полный Pipeline
Соединяем препроцессор и модель в **один объект**. У него те же `.fit()` и `.predict()`, что у обычной модели — но внутри он сам делает все шаги по порядку.

> 🔒 **Самое важное:** Pipeline считает медиану и масштаб **только по train**. Тест остаётся нетронутым — нет **утечки данных**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = Pipeline([
    ('prep',   preprocess),                                       # шаг 1: подготовка
    ('forest', RandomForestClassifier(n_estimators=100, random_state=42)),  # шаг 2: модель
])

model.fit(X_train, y_train)   # одна строка делает ВСЁ по порядку
acc = accuracy_score(y_test, model.predict(X_test))
print(f'Точность на TEST: {acc:.1%}')
print('Заметь: мы НИ РАЗУ не писали fillna или get_dummies вручную!')

### 📝 Задание 1 — зачем нужна подготовка
Что будет, если подать модели **сырые данные с текстом** (`sex='male'`, `embarked='S'`) без кодирования? Запусти ячейку — увидишь ошибку. Это показывает, **зачем** нужен шаг подготовки внутри Pipeline: модель понимает только числа.

*Рабочий пайплайн выше от этого не пострадает — здесь мы просто эксперимент.*

In [ ]:
# Модель БЕЗ подготовки — подаём сырые данные, где есть текст
try:
    raw_model = RandomForestClassifier(random_state=42)
    raw_model.fit(X_train, y_train)   # X_train содержит 'male'/'female', 'S'/'C'/'Q'
    print('Обучилось (не ожидали)')
except Exception as e:
    print('Ошибка (и это ожидаемо):', type(e).__name__)
    print('Модель не понимает текст напрямую — его надо закодировать.')
    print('Именно это делает ColumnTransformer внутри нашего Pipeline. ✅')

## Шаг 3 · Честная оценка — кросс-валидация
Одна проверка на test — это как одна контрольная: могло «повезти» с разбиением. **Кросс-валидация** проверяет модель 5 раз на разных частях данных и даёт **среднее ± разброс**.

In [ ]:
from sklearn.model_selection import cross_val_score

# cv=5 -> данные делятся на 5 частей, 5 раз обучаем и проверяем
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print('Точность на 5 фолдах:', [f'{s:.0%}' for s in scores])
print(f'Среднее: {scores.mean():.1%}  ±  {scores.std():.1%}')

✍️ **Ответь словами.** Числа на 5 фолдах разные. Почему среднее по ним надёжнее, чем одна проверка?

*Твой ответ:* …

### 📝 Задание 2 — улучши модель
Меняем **только последний шаг** конвейера — препроцессинг переиспользуем. Попробуй разные параметры и постарайся поднять среднюю точность.

👉 Меняй `n_estimators` и `max_depth`, запускай снова.

In [ ]:
my_model = Pipeline([
    ('prep', preprocess),
    ('forest', RandomForestClassifier(
        n_estimators=300,   # 👈 меняй
        max_depth=6,        # 👈 меняй (попробуй 4, 6, None)
        random_state=42)),
])

s = cross_val_score(my_model, X, y, cv=5)
print(f'Мой результат: {s.mean():.1%} ± {s.std():.1%}')
# 👉 запиши свой лучший результат в лидерборд на доске!

## ✅ Проверь себя
1. Зачем числам и категориям нужна разная обработка?
2. Как Pipeline защищает от утечки данных?
3. Почему кросс-валидация честнее одной проверки?

<details><summary>Показать ответы</summary>

1. Числа масштабируют, категории превращают в 0/1 — операции разные.
2. Он считает подготовку только по train внутри каждой проверки — тест не «подсматривают».
3. Проверяет модель несколько раз на разных частях и усредняет — меньше зависит от удачи.
</details>

## 🏁 Финальное задание (3 уровня)
**Базовый.** Собери Pipeline (`ColumnTransformer` + модель) и выведи `cross_val_score` как mean ± std.

**Средний.** Добавь или убери признак из `num_features`/`cat_features` — изменилась ли точность?

**Продвинутый.** Подбери параметры через `GridSearchCV` поверх Pipeline.

Пиши код ниже 👇

In [ ]:
# Базовый (стартер) — уже работает
print(f'Базовый: {cross_val_score(model, X, y, cv=5).mean():.1%}')

# Продвинутый (раскомментируй):
# from sklearn.model_selection import GridSearchCV
# grid = {'forest__n_estimators': [100, 300], 'forest__max_depth': [4, 6, None]}
# search = GridSearchCV(model, grid, cv=5)
# search.fit(X, y)
# print('Лучшие параметры:', search.best_params_)
# print(f'Лучшая точность: {search.best_score_:.1%}')

---
### 🎉 Готово!
Теперь у тебя есть собственный конвейер: сырые данные входят — предсказание выходит. **С этого урока любой твой проект собирается как Pipeline.** На следующем уроке узнаем, когда точности недостаточно, и познакомимся с матрицей ошибок.